<table align="left"><tr><td>
<a href="https://colab.research.google.com/github/kikim6114/nlp2026/blob/main/tutors_codes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="코랩에서 실행하기"/></a>
</td></tr></table>

In [ ]:
#  Colab이나 Kaggle을 사용하는 경우가 아니면, 이 cell을 모두 주석화하여 skip할 것.
%cd nlp2026

### tutorscode-1
#### `torch.nn.utils.rnn.pad_sequence`

In [7]:
import torch
a = torch.ones(25, 300)
b = torch.ones(22, 300)
c = torch.ones(15, 300)
inputs = torch.nn.utils.rnn.pad_sequence([a, b, c])
inputs.size()

torch.Size([25, 3, 300])

In [8]:
print(inputs)

tensor([[[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.]],

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.]],

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.]],

        ...,

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]])


In [9]:
inputs = torch.nn.utils.rnn.pad_sequence([a, b, c], batch_first=True, padding_value=0)
inputs.size()

torch.Size([3, 25, 300])

### tutorscode-2
#### `broadcasting`

크기 (seq_length, 1)인 행렬을 (d_model,) 벡터로 나누면 어떻게 처리되는가?
#### broadcasting이 하는 일
1. (d_model,) => (1, d_model) 로 확장
2. (seq_length, 1) => 각 행을 복사하여 (seq_length, d_model)이 된다
3. (1, d_model) => 각 열을 복사하여 (seq_length, d_model)이 된다.
4. 이제 같은 크기의 두 행렬에 대하여 요소별(elment-wise) 나누기 연산을 수행한다.

In [36]:
import numpy as np

seq_length = 64
d_model = 128

# (64, 1) 벡터
pos = np.arange(seq_length)[:, np.newaxis] 

# (128,) 벡터
div = np.power(10000, np.arange(d_model) / d_model)

# (64, 1) / (128,) -> (64, 128)
result = pos / div

print(f"Position shape: {pos.shape}")
print(f"Divisor shape: {div.shape}")
print(f"Result shape: {result.shape}")

Position shape: (64, 1)
Divisor shape: (128,)
Result shape: (64, 128)


분자, 분모를 서로 바꾼 경우

In [37]:
result = div / pos
print(f"Position shape: {pos.shape}")
print(f"Divisor shape: {div.shape}")
print(f"Result shape: {result.shape}")

Position shape: (64, 1)
Divisor shape: (128,)
Result shape: (64, 128)


C:\Users\kikim\AppData\Local\Temp\ipykernel_10288\2375058565.py:1: RuntimeWarning: divide by zero encountered in divide
  result = div / pos


#### [주의]
분자 분모의 rank가 같은 경우, broadcasting이 사용되지 않으므로, 요소별 나눗셈이 될 수 있록 분자,분모의 shape가 같아야 한다.

In [35]:
pos = pos.squeeze()
result = pos / div
print(f"Position shape: {pos.shape}")
print(f"Divisor shape: {div.shape}")
print(f"Result shape: {result.shape}")

Position shape: (128,)
Divisor shape: (128,)
Result shape: (128,)


### tutorscode-3
#### `torch.Tensor.expand`
- 텐서의 차원 크기가 1인 차원을 원하는 크기로 늘려주는(브로드캐스팅) 함수
- 텐서를 확장해도 새로운 메모리가 할당되지는 않고 기존 텐서에 대한 새로운 뷰만 생성
- 주로 신경망에서 특정 차원의 크기를 맞춰주기 위한 브로드캐스팅이 필요할 때 사용됨

In [38]:
x = torch.tensor([[1], [2], [3]])
x.size()

torch.Size([3, 1])

In [10]:
x.expand(3, 4)

tensor([[1, 1, 1, 1],
        [2, 2, 2, 2],
        [3, 3, 3, 3]])

In [11]:
x.expand(-1, 4)   # -1 해당 차원은 변경하지 않는다는 뜻

tensor([[1, 1, 1, 1],
        [2, 2, 2, 2],
        [3, 3, 3, 3]])

### tutorscode-4
#### `torch.Tensor.unsqueeze` or `torch.unsqueeze`
지정된 위치에 크기가 1인 차원이 삽입된 새로운 텐서를 반환.

In [44]:
x = torch.tensor([1, 2, 3, 4])
print(x.shape)
print(torch.unsqueeze(x, 0).shape)
print(torch.unsqueeze(x, 1).shape)

torch.Size([4])
torch.Size([1, 4])
torch.Size([4, 1])


In [48]:
inputs = torch.tensor([[1, 2, 0], [4, 0, 0]])
print(inputs)
print(inputs.shape)
print(inputs.unsqueeze(1).shape)
print(inputs.unsqueeze(1).expand(2, 3, 3,).shape)
print(inputs.unsqueeze(1).expand(2, 3, 3,))

tensor([[1, 2, 0],
        [4, 0, 0]])
torch.Size([2, 3])
torch.Size([2, 1, 3])
torch.Size([2, 3, 3])
tensor([[[1, 2, 0],
         [1, 2, 0],
         [1, 2, 0]],

        [[4, 0, 0],
         [4, 0, 0],
         [4, 0, 0]]])
